# Introdução à Aprendizagem Automática 2025/2026

## TP08: Feature Selection, Dimensionality Reduction and Model Tuning
*A Machine Learning Tutorial by Andre Falcao (DI/FCUL 2020-2022), 
*revised by Docentes MC (DI/FCUL 2022-26)*

### Summary

1. Feature selection
    1. Using correlation
    2. Using stepwise methods
2. Principal Components analysis
    1. Linear PCA
3. Model Tuning


## 1. Feature selection

### 1.1 Correlation

We are going to start by getting our favourite libraries and our dataset arranged for Binary Classification

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
#from sklearn import tree
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import f1_score #, confusion_matrix
from sklearn.metrics import r2_score #, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

Now get the diabetes dataset

In [2]:
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()

# Convert to a pandas dataframe
df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
# Add the target variable to the dataframe
df['target'] = diabetes.target

Let's make a simple evaluation function running 2 regression algorithms and producing the R2 for each

In [3]:
# train-test split
train, test = train_test_split(df, test_size=0.2, random_state=0)

def naive_model_testing(train, test):
    
    #test 2 models, DTs and LR, and print out the results
    dtr= DecisionTreeRegressor(max_depth=5)
    dtr.fit(train.drop('target', axis=1), train['target'])

    lmr=LinearRegression()
    lmr.fit(train.drop('target', axis=1), train['target'])

   # rf_preds=rfr.predict(X_test)
    dt_preds=dtr.predict(test.drop('target', axis=1))
    lr_preds=lmr.predict(test.drop('target', axis=1))

   # print("RVE RFs: %7.4f" % explained_variance_score(y_test, rf_preds))
    print("R2 Decision Tree Regression: %7.4f" % r2_score(test['target'], dt_preds))
    print("R2 Linear Regression: %7.4f" % r2_score(test['target'], lr_preds))

naive_model_testing(train, test)


R2 Decision Tree Regression:  0.0548
R2 Linear Regression:  0.3322


### Correlation 

As a first exercise we are going to use the Spearman correlation

In [4]:
spear = df.corr(method='spearman')
spear

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
age,1.000000,0.177463,0.200554,0.350859,0.262524,0.221711,-0.106973,0.221017,0.265176,0.296235,0.197822
sex,0.177463,1.000000,0.098079,0.261508,0.027790,0.134695,-0.394584,0.337524,0.174625,0.203277,0.037401
bmi,0.200554,0.098079,1.000000,0.397985,0.287829,0.295494,-0.371172,0.459068,0.491609,0.384664,0.561382
bp,0.350859,0.261508,0.397985,1.000000,0.275224,0.205638,-0.191033,0.280799,0.396071,0.381219,0.416241
s1,0.262524,0.027790,0.287829,0.275224,1.000000,0.878793,0.015308,0.520674,0.512864,0.332173,0.232429
s2,0.221711,0.134695,0.295494,0.205638,0.878793,1.000000,-0.197435,0.652283,0.349947,0.286483,0.195834
s3,-0.106973,-0.394584,-0.371172,-0.191033,0.015308,-0.197435,1.000000,-0.789694,-0.450420,-0.290863,-0.410022
s4,0.221017,0.337524,0.459068,0.280799,0.520674,0.652283,-0.789694,1.000000,0.640390,0.413700,0.448931
s5,0.265176,0.174625,0.491609,0.396071,0.512864,0.349947,-0.450420,0.640390,1.000000,0.453023,0.589416
s6,0.296235,0.203277,0.384664,0.381219,0.332173,0.286483,-0.290863,0.413700,0.453023,1.000000,0.350792


### Exercise 1

1. Identify the Top 5 most correlated variables to the y
2. Check if this variable selection is capable of better results than using all variables

In [ ]:
# Exercise 1.1
# bmi, bp, s4, s5, s6


In [12]:
# Exercise 1.2
df_subset = df[['bmi', 'bp', 's4', 's5', 's6', 'target']]
train_subset, test_subset = train_test_split(df_subset, test_size=0.2, random_state=0)
naive_model_testing(train_subset, test_subset)

R2 Decision Tree Regression:  0.0543
R2 Linear Regression:  0.3363


In [ ]:
# Exercise 1.extra
# run previous cell multiple times and observe that DTs score may change.  Why? 




### 1.2. Stepwise Feature selection

Here we are going to use the [Sequential Feature Selector form scikit](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html#sklearn.feature_selection.SequentialFeatureSelector) which takes any supervised method and runs the forward or the backward methods for defining the most relevant variables

By default it uses the forward method and we are going to select the best 5 features

In [13]:
from sklearn.feature_selection import SequentialFeatureSelector

N, M = train.shape
M = M-1  # to have the data shape without target
print('shape: ', N, 'x', M)

shape:  353 x 10


In [14]:
#using linear regression for sequential feature selection
lmr=LinearRegression()
sfs = SequentialFeatureSelector(lmr, n_features_to_select=5).set_output(transform="pandas")
sfs.fit(train.drop('target', axis=1), train['target'])

#get the relevant columns
features=sfs.get_support()
Features_selected =np.arange(M)[features]
print("The features selected are columns: ", Features_selected)

n_train=sfs.transform(train.drop('target', axis=1))
n_test=sfs.transform(test.drop('target', axis=1))

n_train['target'] = train['target']  # form the train dataframe to pass to the metrics function
n_test['target'] = test['target']  # form the test dataframe to pass to the metrics function

naive_model_testing(n_train, n_test)

The features selected are columns:  [1 2 3 4 8]
R2 Decision Tree Regression:  0.0177
R2 Linear Regression:  0.3120


### Exercise 2

1. Run forward sequential fitting for a decision tree with max_depth=3

2. Change the direction to "backward"

In [17]:
#Exercise 2.1
dt = DecisionTreeRegressor(max_depth=3)
sfs_forward = SequentialFeatureSelector(dt, n_features_to_select=5, direction='forward').set_output(transform="pandas")
sfs_forward.fit(train.drop('target', axis=1), train['target'])
print("Forward selection:", sfs_forward.get_feature_names_out())

Forward selection: ['age' 'sex' 'bmi' 's4' 's5']


In [18]:
#Exercise 2.2
sfs_backward = SequentialFeatureSelector(dt, n_features_to_select=5, direction='backward').set_output(transform="pandas")
sfs_backward.fit(train.drop('target', axis=1), train['target'])
print("Backward selection:", sfs_backward.get_feature_names_out())



Backward selection: ['sex' 'bmi' 's3' 's4' 's5']


## 2. Principal Components Analysis

We are now going to use the [PCA module](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) from scikit-learn

First let's just find a 2D projection of our data (remember to use only the training set)


In [20]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3).set_output(transform="pandas") #finding the two best PCs
pca.fit(train.drop('target', axis=1))
tve=0 #total variance explained
for i, ve in enumerate(pca.explained_variance_ratio_):
    tve+=ve
    print("PC%d - Variance explained: %7.4f - Total Variance: %7.4f" % (i, ve, tve) )
print()
print("Actual Eigenvalues:", pca.singular_values_)
for i,comp in enumerate(pca.components_):
    print("PC",i, "-->", comp)
    

PC0 - Variance explained:  0.4154 - Total Variance:  0.4154
PC1 - Variance explained:  0.1402 - Total Variance:  0.5556
PC2 - Variance explained:  0.1154 - Total Variance:  0.6710

Actual Eigenvalues: [1.83741142 1.06757328 0.96856811]
PC 0 --> [ 0.23407134  0.17993547  0.31108607  0.27430238  0.33194466  0.33280716
 -0.28414879  0.42177706  0.38377357  0.33756638]
PC 1 --> [ 0.06064859 -0.37808356 -0.17547633 -0.15725858  0.57785175  0.47586989
  0.47371054 -0.04251347 -0.03067198 -0.10167466]
PC 2 --> [ 0.53534561 -0.06201051  0.11511253  0.48829067 -0.04895193 -0.24709701
  0.4154025  -0.39685378  0.04023075  0.25051387]


### Exercise 3

1. Interpret the results above. 
   
2. What is the meaning of the PC vectors?

In [ ]:
# Exercise 3.1 



# Exercise 3.2 



Now let's project the data using the principal components defined and use them for regression

In [ ]:
n_train=pca.transform(train.drop('target', axis=1))
n_test=pca.transform(test.drop('target', axis=1))
n_train['target'] = train['target']  # form the train dataframe to pass to the metrics function
n_test['target'] = test['target']  # form the test dataframe to pass to the metrics function
naive_model_testing(n_train, n_test)

quite poor results as expected

### A graphical view illustrated with binary classification data

We consider now the same data as a classification problem, assuming that patients with a target value of 250 or more means they have diabetes and with less that 250 they don't

In [ ]:
# with binary classified instances
# target values of 250 or more indicate diabetes
yc_diabetes=np.array([int(i>=250) for i in diabetes.target]) # to be used in graphics ahead

X_train, X_test, y_train, y_test = train_test_split(diabetes.data, yc_diabetes, test_size=0.2, random_state=23)
print("training set patients with target value >=250: ", (y_train).sum())
print("training set patients with target value <250: ", len(y_train) - (y_train).sum())

Let's plot the projection in 2 components 

In [ ]:
pca = PCA(n_components=2) #finding the two best PCs
pca.fit(X_train)
nX_train=pca.transform(X_train)
nX_test=pca.transform(X_test)
colors=np.array(["tab:blue", "tab:orange"])[y_train]
plt.scatter(nX_train[:,0], nX_train[:,1], c=colors)
plt.show()

also as a classification problem we can see it is hard to discriminate the two classes using only the two PCs

## 3. Model Tuning

For this example we are going to use Decision Tree Classifiers, but any model learned so far can be used

We are going to use first [Scikit-Learn's GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html), an implementation of extensive parameter search. In its basic form it just requires:
* a bare bones model constructor 
* a dictionary containing the parameters to search for. The keys of the dictionary should correspond to the parameter to test and the values to a list of possible values to test
* a scoring function defining what is the criterion to select and rank the best models
* GridSearchCV uses by default 5-Fold Cross validation, but other validation criteria can be used

The result of GridSearchCV is a structure that contains the fitted models that can then be used for learning and application

Tet's try it with the max_depth, min_samples_split and ccp_alpha (a regularization parameter) values for classification

In [ ]:
from time import time
#from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import scipy.stats as stats

#make the dictionary with the testing parameters
#gammas = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7]
#Cs = [1, 10, 100, 1e3, 1e4, 1e5]
#param_grid = {'gamma': gammas, 'C': Cs}
depths = [3, 5, 10, 15]
m_sampl_split = [2, 5, 9]
prune_a = [0.0, 0.0001, 0.001, 0.01]
param_grid = {'max_depth': depths, 'min_samples_split': m_sampl_split, 'ccp_alpha': prune_a}

#define the model and do the grid search
#clf = SVC() # RBF (Gaussian) by default
clf = DecisionTreeClassifier(criterion='log_loss', random_state=23)
gs = GridSearchCV(estimator=clf, param_grid=param_grid, scoring="f1")

start = time()
gs=gs.fit(X_train, y_train)
print(
    'GridSearchCV took %.2f seconds for %d candidate parameter settings.'
    % ((time() - start), len(gs.cv_results_['params']))
)

Let's identify the best element parameters [best according to the scoring function, in this case it is the F1 score]

In [ ]:
#print('best gamma: %7.4f' % gs.best_estimator_.gamma)
#print('best C: %3.2f' %  gs.best_estimator_.C)
print('best maximum depth: %2.0f' % gs.best_estimator_.max_depth)
print('best minimum samples to split a node: %2.0f' %  gs.best_estimator_.min_samples_split)
print('best minimal cost pruning parameter: %1.4f' % gs.best_estimator_.ccp_alpha)

Just for sake of completion, we can use the best estimator model (the one with the optimized parameters) for prediction on the test set.

In [ ]:
preds=gs.best_estimator_.predict(X_test)
print('F1 : %7.4f' % f1_score(y_test, preds))
print('number of leaves:', gs.best_estimator_.get_n_leaves())

GridSearchCV gives you a number of statistics on the tests it runs:

In [ ]:
for i in gs.cv_results_.keys(): print(i)

We can print the results in a nice Pandas Data Frame

In [ ]:
grid_res = pd.DataFrame(gs.cv_results_)
grid_res.sort_values(by=['rank_test_score'], ascending=True, inplace=True) #sort the tested models by score
grid_res[['params', 'rank_test_score', 'mean_test_score', 'std_test_score', 'mean_fit_time', 'std_fit_time']] #show only mean and std of the test score

we can check if the 2nd best model produces different results 

In [ ]:
print('max_depth:', grid_res['param_max_depth'].iat[1],
      'min_samples_split:', grid_res['param_min_samples_split'].iat[1],
      'ccp_alpha:', '{:.2e}'.format(grid_res['param_ccp_alpha'].iat[1]))
clf = DecisionTreeClassifier(criterion='log_loss', random_state=23,
                             max_depth=grid_res['param_max_depth'].iat[1],
                             min_samples_split=grid_res['param_min_samples_split'].iat[1],
                             ccp_alpha=grid_res['param_ccp_alpha'].iat[1])
clf.fit(X_train, y_train)
preds=clf.predict(X_test)
print('F1 : %7.4f' % f1_score(y_test, preds))
print('number of leaves:', clf.get_n_leaves())

Let's try now the RandomizedSearchCV and compare to the previous one.

In [ ]:
# configure randomized search (by default also 5-fold CV)
# notice the loguniform distributions

param_dist = {
#    'C': stats.loguniform(1, 1e5),
#    'gamma': stats.loguniform(1e-7, 1e-1),
    'max_depth': stats.randint(3, 16),
    'min_samples_split': stats.randint(2, 10),
    'ccp_alpha': stats.loguniform(1e-5, 0.01)
}

n_iter_search = 15
rs = RandomizedSearchCV(
    clf, param_distributions=param_dist, n_iter=n_iter_search
)

start = time()
rs = rs.fit(X_train, y_train)
print(
    'RandomizedSearchCV took %.2f seconds for %d candidates parameter settings'
    % ((time() - start), n_iter_search)
)

In [ ]:
print('best maximum depth: %2.0f' % rs.best_estimator_.max_depth)
print('best minimum samples to split a node: %2.0f' %  rs.best_estimator_.min_samples_split)
print('best minimal cost pruning parameter: %1.4f' % rs.best_estimator_.ccp_alpha)

Now we can use the best estimator model (the one with the optimized parameters) for prediction

In [ ]:
rs1 = rs.best_estimator_
rs1.fit(X_train, y_train)
preds=rs1.predict(X_test)
print('F1 : %7.4f' % f1_score(y_test, preds))
print('number of leaves:', rs1.get_n_leaves())

In [ ]:
rand_res = pd.DataFrame(rs.cv_results_)
rand_res.sort_values(by=['rank_test_score'], ascending= True, inplace=True) #sort the tested models by score
rand_res[['params', 'rank_test_score', 'mean_test_score', 'std_test_score', 'mean_fit_time', 'std_fit_time']] #show only mean and std of the test score

checking the 2nd best model 

In [ ]:
print('max_depth:', rand_res['param_max_depth'].iat[1],
      ', min_samples_split:', rand_res['param_min_samples_split'].iat[1],
      ', ccp_alpha:', '{:.2e}'.format(rand_res['param_ccp_alpha'].iat[1]))
clf = DecisionTreeClassifier(criterion='log_loss', random_state=23,
                             max_depth=rand_res['param_max_depth'].iat[1],
                             min_samples_split=rand_res['param_min_samples_split'].iat[1],
                             ccp_alpha=rand_res['param_ccp_alpha'].iat[1])
clf.fit(X_train, y_train)
preds=clf.predict(X_test)
print('F1 : %7.4f' % f1_score(y_test, preds))
print('number of leaves:', clf.get_n_leaves())


### Exercise 4
1. Discuss the values above in terms of coherency of the parameters found. Do you find a pattern in the best values for max_dept and ccp_alpha?
2. Compare the first 3 models results using the testing set and discuss your findings [Optional]


In [ ]:
# Exercise 4.1



In [ ]:
# Exercise 4.2



In [ ]:
# Comments on results of Exercise 4.2

